# Train Tallie 1.0

The steps and settings used to train the released model:
Qwen3-1.7B → 20 of 28 layers, bidirectional attention, mention / relevance / opinion / type heads, LoRA (r=16), 3 epochs, relevance cutoff tuned on validation.

**Needs, in Google Drive under `MyDrive/Tallie/`:**
- `data/labeled/labels_clean.jsonl`: labeled comments (format produced by `scripts/label_teacher.py`)
- `data/gold_test.jsonl`: the hand-labeled test set (`scripts/fetch_gold_text.py` builds it)

**GPU:** L4 or A100 (Runtime → Change runtime type). Keep at least 8 GB free in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, sys
os.chdir("/content/drive/MyDrive/Tallie"); print("working in", os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
need = ["data/labeled/labels_clean.jsonl", "data/gold_test.jsonl"]
missing = [p for p in need if not os.path.exists(p)]
print("missing files:", missing or "none")
assert not missing
!pip -q install -U "transformers>=4.55" "peft>=0.15" accelerate
!pip -q uninstall -y torchao
!mkdir -p /content/repo/tallie /content/repo/scripts

## Code

In [ ]:
%%writefile /content/repo/tallie/__init__.py
"""Tallie: reads the thread so you don't have to."""
from .pipeline import Tallie
from .model import TallieModel, TYPES, POLARITY

__version__ = "1.0.0"

In [ ]:
%%writefile /content/repo/tallie/data.py
"""Tokenization + label alignment for Tallie."""
import json
import torch
from torch.utils.data import Dataset

from .model import BIO_O, BIO_B, BIO_I, TYPES

TYPE_ID = {t: i for i, t in enumerate(TYPES)}

QUERY_TAG, TEXT_TAG = "Query: ", "\nText: "


def build_input(tokenizer, query, text, max_len):
    prefix = QUERY_TAG + query + TEXT_TAG
    enc = tokenizer(prefix + text, return_offsets_mapping=True, truncation=True,
                    max_length=max_len, add_special_tokens=False)
    offs = enc["offset_mapping"]
    q_lo, q_hi = len(QUERY_TAG), len(QUERY_TAG) + len(query)
    base = len(prefix)
    query_mask = [int(s < q_hi and e > q_lo) for s, e in offs]
    text_mask = [int(e > base) for s, e in offs]
    return enc["input_ids"], offs, query_mask, text_mask, base


def char_span_to_tokens(offs, base, s, e):
    s, e = s + base, e + base
    toks = [i for i, (ts, te) in enumerate(offs) if te > s and ts < e]
    return (toks[0], toks[-1]) if toks else None


class TallieDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=384, limit=0):
        self.rows = [json.loads(l) for l in open(path, encoding="utf-8")]
        if limit:
            self.rows = self.rows[:limit]
        self.tok, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        ids, offs, qmask, tmask, base = build_input(self.tok, r["query"], r["text"], self.max_len)
        bio = [BIO_O if t else -100 for t in tmask]
        bio = [b if q == 0 else BIO_O for b, q in zip(bio, qmask)]
        spans, rel, pol, typ = [], [], [], []
        for sp in r["spans"]:
            te = char_span_to_tokens(offs, base, sp["start"], sp["end"])
            if te is None:
                continue  # truncated away
            a, b = te
            bio[a] = BIO_B
            for k in range(a + 1, b + 1):
                bio[k] = BIO_I
            spans.append([a, b]); rel.append(sp["relevant"]); pol.append(sp["polarity"])
            typ.append(TYPE_ID.get(sp.get("type"), -100))
        return {"input_ids": ids, "query_mask": qmask, "bio": bio,
                "spans": spans, "rel": rel, "pol": pol, "typ": typ}


def collate(batch, pad_id):
    L = max(len(b["input_ids"]) for b in batch)
    S = max(1, max(len(b["spans"]) for b in batch))
    B = len(batch)
    out = {
        "input_ids": torch.full((B, L), pad_id, dtype=torch.long),
        "attention_mask": torch.zeros(B, L, dtype=torch.long),
        "query_mask": torch.zeros(B, L, dtype=torch.long),
        "bio_labels": torch.full((B, L), -100, dtype=torch.long),
        "span_idx": torch.zeros(B, S, 2, dtype=torch.long),
        "span_valid": torch.zeros(B, S, dtype=torch.long),
        "rel_labels": torch.zeros(B, S, dtype=torch.long),
        "pol_labels": torch.zeros(B, S, dtype=torch.long),
        "type_labels": torch.full((B, S), -100, dtype=torch.long),
    }
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        out["input_ids"][i, :n] = torch.tensor(b["input_ids"])
        out["attention_mask"][i, :n] = 1
        out["query_mask"][i, :n] = torch.tensor(b["query_mask"])
        out["bio_labels"][i, :n] = torch.tensor(b["bio"])
        k = len(b["spans"])
        if k:
            out["span_idx"][i, :k] = torch.tensor(b["spans"])
            out["span_valid"][i, :k] = 1
            out["rel_labels"][i, :k] = torch.tensor(b["rel"])
            out["pol_labels"][i, :k] = torch.tensor(b["pol"])
            if b.get("typ"):
                out["type_labels"][i, :k] = torch.tensor(b["typ"])
    return out

In [ ]:
%%writefile /content/repo/tallie/filters.py
"""Rules for what counts as a *named item*. Used both when building training spans
and when ranking results, so junk like "books", "it", "The" or "," never becomes an item."""
import re

GENERIC = set("""
a an the it its this that these those one ones they them he she him her his i me my we us you your
book books novel novels story stories series trilogy volume author authors writer writers
movie movies film films show shows tv episode episodes season seasons documentary documentaries
game games video videos podcast podcasts episode album albums song songs music band bands track tracks
software tool tools app apps program programs language languages library libraries framework editor
fiction nonfiction non-fiction sci-fi scifi science fantasy horror comedy drama romance thriller history
philosophy literature poetry biography biographies memoir textbook textbooks classic classics
anime manga comic comics cartoon cartoons article articles essay essays paper papers blog blogs
something anything everything nothing thing things stuff lot lots part
""".split())

_ALNUM = re.compile(r"[A-Za-z0-9]")


def is_valid_item(text: str) -> bool:
    t = text.strip().strip("\"'“”‘’*_()[]:;,.!?").strip()
    if t in {"C", "R", "D", "J"}:   # single-letter programming languages
        return True
    if len(t) < 2 or not _ALNUM.search(t):
        return False
    low = t.lower()
    if low in GENERIC:
        return False
    words = re.findall(r"[a-z0-9]+", low)
    if words and all(w in GENERIC for w in words):  # e.g. "the books", "sci fi movies"
        return False
    return True


def find_all(text: str, needle: str):
    """Whole-word occurrences. Case-insensitive fallback only for longer names
    (so the novel "IT" never matches every "it")."""
    pat = r"(?<![\w])" + re.escape(needle) + r"(?![\w])"
    hits = list(re.finditer(pat, text))
    if not hits and len(needle) >= 4:
        hits = list(re.finditer(pat, text, flags=re.I))
    return hits


def clean_span(text: str, s: int, e: int):
    """Fix model spans that start/end mid-word ('loat', 'L' from 'Luddites') by snapping to
    whole words, and drop spans that sit inside a URL. Returns (s, e) or None."""
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    if s >= e:
        return None
    while s > 0 and text[s - 1].isalnum():
        s -= 1
    while e < len(text) and text[e].isalnum():
        e += 1
    a = text.rfind(" ", 0, s) + 1
    b = text.find(" ", e)
    run = text[a: b if b != -1 else len(text)]
    if "://" in run or "www." in run or run.count("/") >= 2:
        return None
    return s, e

In [ ]:
%%writefile /content/repo/tallie/infer.py
"""Inference, ranking (the counting part), save/load and the slim export."""
import json, re
from collections import Counter, defaultdict
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

from .data import build_input, collate
from .model import TallieModel, POLARITY, TYPES, amp_dtype
from .filters import is_valid_item, clean_span


# ---------------------------------------------------------------------------- inference
def chunk(text, max_chars=1500):
    if len(text) <= max_chars:
        return [(0, text)]
    out, i = [], 0
    while i < len(text):
        j = min(len(text), i + max_chars)
        if j < len(text):
            k = text.rfind(". ", i + 200, j)
            j = k + 1 if k > 0 else j
        out.append((i, text[i:j]))
        i = j
    return out


@torch.no_grad()
def extract(model, tok, query, texts, batch_size=16, max_len=384, threshold=0.5):
    """Returns, for each text, the list of items that satisfy `query`."""
    model.eval()
    dev = next(model.parameters()).device
    pieces = [(ti, off, c) for ti, t in enumerate(texts) for off, c in chunk(t)]
    pieces.sort(key=lambda p: len(p[2]))  # similar lengths per batch = less padding, ~2x faster
    results = [[] for _ in texts]
    for b0 in range(0, len(pieces), batch_size):
        batch = pieces[b0:b0 + batch_size]
        feats, metas = [], []
        for ti, off, c in batch:
            ids, offs, qm, tm, base = build_input(tok, query, c, max_len)
            feats.append({"input_ids": ids, "query_mask": qm, "bio": [0] * len(ids),
                          "spans": [], "rel": [], "pol": []})
            metas.append((ti, off, c, offs, tm, base))
        x = collate(feats, tok.pad_token_id)
        x = {k: v.to(dev) for k, v in x.items()}
        with torch.autocast("cuda", dtype=amp_dtype(), enabled=dev.type == "cuda"):
            h = model.encode(x["input_ids"], x["attention_mask"])
        tags = model.mention_head(h).argmax(-1).tolist()
        span_lists = [model.decode_bio(tg[:len(f["input_ids"])], m[4])
                      for tg, f, m in zip(tags, feats, metas)]
        S = max(1, max(len(s) for s in span_lists))
        idx = torch.zeros(len(batch), S, 2, dtype=torch.long, device=dev)
        for i, sl in enumerate(span_lists):
            if sl:
                idx[i, :len(sl)] = torch.tensor(sl, device=dev)
        rel, pol, _ = model.heads(h, x["query_mask"], idx)
        rel, pol = torch.sigmoid(rel).tolist(), pol.softmax(-1).tolist()
        for i, sl in enumerate(span_lists):
            ti, off, c, offs, tm, base = metas[i]
            for j, (a, b) in enumerate(sl):
                fixed = clean_span(c, max(offs[a][0] - base, 0), offs[b][1] - base)
                if fixed is None:
                    continue
                cs, ce = fixed
                surface = c[cs:ce]
                if rel[i][j] >= threshold:
                    p = max(range(4), key=lambda k: pol[i][j][k])
                    results[ti].append({"text": surface, "start": off + cs, "end": off + ce,
                                        "score": round(rel[i][j], 3), "polarity": POLARITY[p]})
    return results


def normalize(name):
    n = name.lower().strip().strip("\"'“”‘’*_()[]:;,.!?")
    n = re.sub(r"^(the|a|an)\s+", "", n)
    return re.sub(r"\s+", " ", n)


def rank(results, top_k=50):
    """The counting step: plain code, exact and free."""
    groups = defaultdict(list)
    for per_text in results:
        for m in per_text:
            if is_valid_item(m["text"]):
                groups[normalize(m["text"])].append(m)
    table = []
    for key, ms in groups.items():
        pol = Counter(m["polarity"] for m in ms)
        table.append({"item": Counter(m["text"] for m in ms).most_common(1)[0][0],
                      "mentions": len(ms),
                      "avg_score": round(sum(m["score"] for m in ms) / len(ms), 3),
                      "polarity": dict(pol)})
    table.sort(key=lambda r: (-r["mentions"], -r["avg_score"]))
    return table[:top_k]


# ---------------------------------------------------------------------------- save / load
def save(model, tok, outdir, extra=None):
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    bb = model.backbone
    if hasattr(bb, "merge_and_unload"):
        bb = bb.merge_and_unload()
    bb.save_pretrained(out / "backbone", safe_serialization=True)
    tok.save_pretrained(out / "backbone")
    heads = {k: v for k, v in model.state_dict().items() if not k.startswith("backbone.")}
    torch.save(heads, out / "heads.pt")
    old = {}
    for name in ("tallie_config.json",):
        if (out / name).exists():
            old = json.loads((out / name).read_text()); break
    cfg = dict(model.cfg, lora_r=0, n_layers_original=model.n_layers_original,
               threshold=getattr(model, "threshold", old.get("threshold", 0.5)))
    cfg.update(extra or {})
    (out / "tallie_config.json").write_text(json.dumps(cfg, indent=2))


def load(outdir, device="cuda", dtype=None):
    out = Path(outdir)
    if dtype is None:
        dtype = amp_dtype() if str(device).startswith("cuda") else torch.float32
    cfg_file = out / "tallie_config.json"
    cfg = json.loads(cfg_file.read_text())
    bb = AutoModel.from_pretrained(out / "backbone", torch_dtype=dtype, attn_implementation="sdpa")
    m = TallieModel(backbone=bb, keep_layers=None, bidirectional=cfg["bidirectional"], lora_r=0,
                     gradient_checkpointing=False, span_dim=cfg.get("span_dim", 512))
    m.n_layers_original = cfg.get("n_layers_original", m.n_layers_original)
    m.threshold = cfg.get("threshold", 0.5)
    heads = torch.load(out / "heads.pt", map_location="cpu")
    if heads.get("vocab_map") is not None:
        m.vocab_map = heads["vocab_map"]
    m.load_state_dict(heads, strict=False)
    tok = AutoTokenizer.from_pretrained(out / "backbone")
    return m.to(device), tok


# ---------------------------------------------------------------------------- slim export
def trim_vocab(model, tok, corpus_texts, extra_ids=()):
    """Shrink the embedding table for English use.
    Keeps EVERY token that decodes to plain ASCII text (all of English, code, numbers, punctuation)
    plus every token seen in `corpus_texts`. Drops tokens for other scripts (Chinese, Arabic, ...).
    (Keeping only corpus tokens would break questions that contain unseen words.)
    Anything not kept maps to one shared 'unknown' row."""
    base = model.backbone.get_base_model() if hasattr(model.backbone, "get_base_model") else model.backbone
    n_vocab = base.embed_tokens.weight.shape[0]
    used = set(extra_ids) | set(tok.all_special_ids)
    for i in range(min(n_vocab, len(tok))):
        piece = tok.decode([i])
        if piece and piece.isascii():
            used.add(i)
    for t in corpus_texts:
        used.update(tok(t, add_special_tokens=False)["input_ids"])
    used.update(range(256))
    if tok.pad_token_id is not None:
        used.add(tok.pad_token_id)
    keep = sorted(i for i in used if i < n_vocab)
    W = base.embed_tokens.weight.data
    newW = torch.cat([W[keep], W[keep].mean(0, keepdim=True)], 0)
    emb = nn.Embedding(newW.shape[0], newW.shape[1], dtype=W.dtype, device=W.device)
    emb.weight.data.copy_(newW)
    base.embed_tokens = emb
    base.config.vocab_size = newW.shape[0]
    vm = torch.full((W.shape[0],), len(keep), dtype=torch.long, device=W.device)
    vm[torch.tensor(keep, device=W.device)] = torch.arange(len(keep), device=W.device)
    model.vocab_map = vm
    for k in ("bos_token_id", "eos_token_id"):  # ids refer to the old vocabulary
        setattr(base.config, k, None)
    return len(keep) + 1


@torch.no_grad()
def extract_all(model, tok, texts, batch_size=32, max_len=384, query="all named items mentioned"):
    """One pass over all texts: every named item with the commenter's opinion and a context window.
    Questions are then answered from this list (see ranking.py) without reading the texts again."""
    model.eval()
    dev = next(model.parameters()).device
    pieces = [(ti, off, c) for ti, t in enumerate(texts) for off, c in chunk(t)]
    pieces.sort(key=lambda p: len(p[2]))
    out = []
    for b0 in range(0, len(pieces), batch_size):
        batch = pieces[b0:b0 + batch_size]
        feats, metas = [], []
        for ti, off, c in batch:
            ids, offs, qm, tm, base = build_input(tok, query, c, max_len)
            feats.append({"input_ids": ids, "query_mask": qm, "bio": [0] * len(ids),
                          "spans": [], "rel": [], "pol": []})
            metas.append((ti, c, offs, tm, base))
        x = {k: v.to(dev) for k, v in collate(feats, tok.pad_token_id).items()}
        with torch.autocast("cuda", dtype=amp_dtype(), enabled=dev.type == "cuda"):
            h = model.encode(x["input_ids"], x["attention_mask"])
        tags = model.mention_head(h).argmax(-1).tolist()
        span_lists = [model.decode_bio(tg[:len(f["input_ids"])], m[3]) for tg, f, m in zip(tags, feats, metas)]
        S = max(1, max(len(s) for s in span_lists))
        idx = torch.zeros(len(batch), S, 2, dtype=torch.long, device=dev)
        for i, sl in enumerate(span_lists):
            if sl:
                idx[i, :len(sl)] = torch.tensor(sl, device=dev)
        _, pol, typ = model.heads(h, x["query_mask"], idx)
        pol = pol.argmax(-1).tolist()
        tprob = typ.softmax(-1)
        tconf, tid = tprob.max(-1)
        tid, tconf = tid.tolist(), tconf.tolist()
        for i, sl in enumerate(span_lists):
            ti, c, offs, tm, base = metas[i]
            for j, (a, b) in enumerate(sl):
                fx = clean_span(c, max(offs[a][0] - base, 0), offs[b][1] - base)
                if fx is None:
                    continue
                s, e = fx
                name = c[s:e].strip("\"'“”‘’")
                if not is_valid_item(name):
                    continue
                out.append({"text": name, "polarity": POLARITY[pol[i][j]], "doc": ti,
                            "type": TYPES[tid[i][j]], "type_conf": round(tconf[i][j], 3),
                            "context": c[max(0, s - 150):e + 150]})
    return out

In [ ]:
%%writefile /content/repo/tallie/model.py
"""
Tallie: a non-generative "query -> list" model built by surgery on a pretrained LLM.

  1. Load Qwen3 WITHOUT its LM head (AutoModel gives the bare transformer).
  2. Prune the top layers (keep the first `keep_layers`).
  3. Switch causal attention to bidirectional, so every token sees the whole input.
  4. Add three new heads that read the final hidden states once:
       mention head   : BIO tag per token  -> where are the named items?
       relevance head : per span, given the query -> does it satisfy the request? (0..1)
       polarity head  : per span -> positive / negative / neutral / mixed
       type head      : per span -> book / movie / tv_show / video_game / ...
No decoding loop: one forward pass produces the whole answer.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

BIO_O, BIO_B, BIO_I = 0, 1, 2
POLARITY = ["positive", "negative", "neutral", "mixed"]
TYPES = ["book", "movie", "tv_show", "video_game", "board_game", "podcast", "music", "software",
         "product", "person", "other"]


def amp_dtype():
    """bf16 on Ampere+ (A100, L4), fp16 on older GPUs (T4), fp32 on CPU."""
    if not torch.cuda.is_available():
        return torch.float32
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def mlp(i, h, o, p=0.1):
    return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, o))


class TallieModel(nn.Module):
    def __init__(self, backbone_name="Qwen/Qwen3-1.7B", keep_layers=20, bidirectional=True,
                 lora_r=16, gradient_checkpointing=True, dtype=None, backbone=None,
                 span_dim=512, bio_weight=3.0, pol_class_weights=None):
        self_bio_weight = bio_weight
        if dtype is None:
            dtype = amp_dtype()
        super().__init__()
        self.bio_weight = self_bio_weight
        self.cfg = dict(backbone_name=backbone_name, keep_layers=keep_layers,
                        bidirectional=bidirectional, lora_r=lora_r, span_dim=span_dim)
        if backbone is None:
            backbone = AutoModel.from_pretrained(backbone_name, torch_dtype=dtype,
                                                 attn_implementation="sdpa")
        self.n_layers_original = len(backbone.layers)

        # --- surgery 1: prune top layers -------------------------------------------------
        if keep_layers and keep_layers < len(backbone.layers):
            backbone.layers = nn.ModuleList(list(backbone.layers)[:keep_layers])
            backbone.config.num_hidden_layers = keep_layers
            if getattr(backbone.config, "layer_types", None):
                backbone.config.layer_types = backbone.config.layer_types[:keep_layers]

        # --- surgery 2: bidirectional attention ------------------------------------------
        self.bidirectional = bidirectional
        if bidirectional:
            for layer in backbone.layers:
                layer.self_attn.is_causal = False

        if gradient_checkpointing:
            backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            backbone.enable_input_require_grads()

        if lora_r:
            from peft import LoraConfig, get_peft_model
            backbone = get_peft_model(backbone, LoraConfig(
                r=lora_r, lora_alpha=2 * lora_r, lora_dropout=0.05, bias="none",
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"]))
            for p in backbone.parameters():  # trainable LoRA weights in fp32 (stable with fp16 AMP)
                if p.requires_grad:
                    p.data = p.data.float()
        self.backbone = backbone

        H = backbone.config.hidden_size
        D = span_dim
        # --- new heads (kept in fp32) ----------------------------------------------------
        self.mention_head = mlp(H, H // 2, 3)
        self.span_proj = mlp(3 * H, D, D)
        self.query_proj = nn.Linear(H, D)
        self.relevance_head = mlp(3 * D, D, 1)
        self.polarity_head = mlp(D, D // 2, len(POLARITY))
        self.type_head = mlp(D, D // 2, len(TYPES))
        # inverse-frequency weights so the polarity head isn't biased toward "positive"
        self.register_buffer("pol_w", torch.tensor(pol_class_weights or [1.0] * len(POLARITY)), persistent=False)
        self.register_buffer("vocab_map", None)  # optional: set by trim_vocab() for the slim export

    # ------------------------------------------------------------------------------------
    def _attn_mask(self, attention_mask, dtype):
        if not self.bidirectional:
            return attention_mask  # model builds the usual causal mask
        # 4D additive mask: every position may attend to every non-pad position.
        B, L = attention_mask.shape
        m = torch.zeros(B, 1, L, L, dtype=dtype, device=attention_mask.device)
        return m.masked_fill(attention_mask[:, None, None, :] == 0, torch.finfo(dtype).min)

    def encode(self, input_ids, attention_mask):
        if self.vocab_map is not None:
            input_ids = self.vocab_map[input_ids]
        emb = self.backbone.get_input_embeddings()
        dtype = emb.weight.dtype
        out = self.backbone(input_ids=input_ids,
                            attention_mask=self._attn_mask(attention_mask, dtype),
                            use_cache=False)
        return out.last_hidden_state.float()

    @staticmethod
    def span_reps(h, span_idx):
        """h: (B,L,H); span_idx: (B,S,2) inclusive token [start,end]. Returns (B,S,3H)."""
        B, L, H = h.shape
        s = span_idx[..., 0].clamp(0, L - 1)
        e = span_idx[..., 1].clamp(0, L - 1)
        gather = lambda idx: torch.gather(h, 1, idx.unsqueeze(-1).expand(-1, -1, H))
        cs = torch.cat([h.new_zeros(B, 1, H), h.cumsum(1)], 1)  # prefix sums
        tot = torch.gather(cs, 1, (e + 1).unsqueeze(-1).expand(-1, -1, H)) - \
              torch.gather(cs, 1, s.unsqueeze(-1).expand(-1, -1, H))
        mean = tot / (e - s + 1).clamp(min=1).unsqueeze(-1).float()
        return torch.cat([gather(s), gather(e), mean], -1)

    def heads(self, h, query_mask, span_idx):
        qm = query_mask.unsqueeze(-1).float()
        q = self.query_proj((h * qm).sum(1) / qm.sum(1).clamp(min=1))         # (B,D)
        sp = self.span_proj(self.span_reps(h, span_idx))                       # (B,S,D)
        qe = q.unsqueeze(1).expand_as(sp)
        rel = self.relevance_head(torch.cat([sp, qe, sp * qe], -1)).squeeze(-1)  # (B,S)
        pol = self.polarity_head(sp)                                             # (B,S,4)
        typ = self.type_head(sp)                                                 # (B,S,11)
        return rel, pol, typ

    def forward(self, input_ids, attention_mask, query_mask, span_idx, span_valid,
                bio_labels=None, rel_labels=None, pol_labels=None, type_labels=None,
                pol_weight=0.5, type_weight=0.5):
        h = self.encode(input_ids, attention_mask)
        bio_logits = self.mention_head(h)
        rel, pol, typ = self.heads(h, query_mask, span_idx)
        out = {"bio_logits": bio_logits, "rel_logits": rel, "pol_logits": pol, "type_logits": typ}
        if bio_labels is not None:
            # names are rare tokens: up-weight B/I so the model doesn't learn "everything is O"
            w = bio_logits.new_tensor([1.0, self.bio_weight, self.bio_weight])
            l_bio = F.cross_entropy(bio_logits.view(-1, 3), bio_labels.view(-1), weight=w,
                                    ignore_index=-100)
            v = span_valid.bool()
            if v.any():
                l_rel = F.binary_cross_entropy_with_logits(rel[v], rel_labels[v].float())
                l_pol = F.cross_entropy(pol[v], pol_labels[v], weight=self.pol_w.to(pol.dtype))
                if type_labels is not None and (type_labels[v] >= 0).any():
                    l_typ = F.cross_entropy(typ[v], type_labels[v], ignore_index=-100)
                else:
                    l_typ = h.sum() * 0
            else:
                l_rel = l_pol = l_typ = h.sum() * 0
            out.update(loss=l_bio + l_rel + pol_weight * l_pol + type_weight * l_typ,
                       l_bio=l_bio.detach(), l_rel=l_rel.detach(), l_pol=l_pol.detach(),
                       l_typ=l_typ.detach())
        return out

    # ------------------------------------------------------------------------------------
    @staticmethod
    def decode_bio(tags, text_mask):
        """tags: list[int] per token. Returns list of (start_tok, end_tok) inclusive."""
        spans, start = [], None
        for i, (t, ok) in enumerate(zip(tags, text_mask)):
            if not ok:
                t = BIO_O
            if t == BIO_B or (t == BIO_I and start is None):
                if start is not None:
                    spans.append((start, i - 1))
                start = i
            elif t == BIO_O and start is not None:
                spans.append((start, i - 1))
                start = None
        if start is not None:
            spans.append((start, len(tags) - 1))
        return spans

    def head_params(self):
        return [p for n, p in self.named_parameters() if not n.startswith("backbone.")]

    def param_report(self):
        bb = self.backbone
        base = bb.get_base_model() if hasattr(bb, "get_base_model") else bb
        emb = base.embed_tokens.weight.numel()
        total_bb = sum(p.numel() for n, p in base.named_parameters() if "lora_" not in n)
        heads = sum(p.numel() for p in self.head_params())
        return {"layers_kept": len(base.layers), "layers_original": self.n_layers_original,
                "embedding_params": emb, "backbone_params": total_bb, "head_params": heads,
                "total_params": total_bb + heads}

In [ ]:
%%writefile /content/repo/tallie/pipeline.py
"""
    from tallie import Tallie
    t = Tallie.load("path/to/model")
    t.index(comments)                         # one pass over your texts (list of strings)
    t.ask("podcasts people recommend")        # instant after indexing
    t.extract(["I loved Dune but Snow Crash bored me."])
"""
import torch

from .infer import load as load_model, extract_all
from .ranking import build_index, answer, make_judge


class Tallie:
    def __init__(self, model, tok, threshold=0.5):
        self.model, self.tok, self.threshold = model, tok, threshold
        self.judge = make_judge(model, tok)
        self._index = None

    @classmethod
    def load(cls, path, device=None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        model, tok = load_model(path, device=device)
        return cls(model, tok, threshold=getattr(model, "threshold", 0.5))

    def index(self, texts, batch_size=16):
        """Read every text once and keep every named item with its type and opinion."""
        self._index = build_index(extract_all(self.model, self.tok, list(texts), batch_size=batch_size))
        return self

    def ask(self, query, top=15):
        if self._index is None:
            raise RuntimeError("call .index(texts) first")
        return answer(query, self._index, top=top, judge=self.judge, judge_threshold=self.threshold)

    def extract(self, texts):
        """Raw output: every named item with its type and the commenter's opinion, per text."""
        return extract_all(self.model, self.tok, list(texts))

In [ ]:
%%writefile /content/repo/tallie/ranking.py
"""
Turning per-comment extractions into ranked answers.

Tallie's model reads each comment once and returns every named item with its type and the
commenter's opinion. A question such as "podcasts people recommend" is answered by
  1. parsing it into a kind of thing (podcast) and an opinion (positive),
  2. keeping the items the model typed as that kind, with that opinion,
  3. tallying across all comments (one vote per comment) and ranking.
Extra topic words ("for beginners", "about space") are checked by the model's relevance head.
"""
import re
from collections import Counter, defaultdict

QUERY_TYPES = [  # words in a question -> wanted type
    ("tv_show", r"\b(tv shows?|television|series|sitcoms?|anime)\b"),
    ("board_game", r"\b(board games?|card games?|tabletop( games?)?)\b"),
    ("video_game", r"\b(video games?|games?)\b"),
    ("podcast", r"\bpodcasts?\b"),
    ("movie", r"\b(movies?|films?|documentar(y|ies))\b"),
    ("book", r"\b(books?|novels?|reads?|reading)\b"),
    ("music", r"\b(albums?|songs?|music|bands?)\b"),
    ("software", r"\b(software|tools?|apps?|programs?|editors?|browsers?|languages?|libraries|frameworks?)\b"),
    ("person", r"\b(authors?|writers?|directors?|people to follow)\b"),
]
TYPE_FAMILY = {"board_game": {"board_game"}, "video_game": {"video_game"},
               "software": {"software", "product"}, "person": {"person"}}
POS_WORDS = r"\b(lik(e|ed)|lov(e|ed)|favou?rites?|best|great|recommend(ed)?|praised?|enjoy(ed)?|good|top)\b"
NEG_WORDS = r"\b(hat(e|ed)|dislik(e|ed)|worst|regret(ted)?|overrated|criticiz(e|ed)|bad|awful|disappointing|complain(ed)?)\b"
STOP = set("""a an the of for about to in on with and or by from that which who people commenters
comments liked loved like love hate hated disliked best worst most favorite favourite recommended
recommend regret regretted listening watching reading read watch play played good great bad
things stuff related everyone anyone some any all mentioned named discussed here
found find think thought say said consider considered really very much more less ones one
would should could get got make made also just""".split())


def _norm(t):
    t = t.lower().strip().strip("\"'“”‘’*_()[]:;,.!?")
    t = re.sub(r"^(the|a|an)\s+", "", t)
    return re.sub(r"[\s\-–:]+", " ", t)


def parse_query(q):
    ql = q.lower()
    want_type = next((t for t, pat in QUERY_TYPES if re.search(pat, ql)), None)
    sentiment = "negative" if re.search(NEG_WORDS, ql) else "positive" if re.search(POS_WORDS, ql) else None
    rest = ql
    for _, pat in QUERY_TYPES:
        rest = re.sub(pat, " ", rest)
    rest = re.sub(POS_WORDS, " ", rest)
    rest = re.sub(NEG_WORDS, " ", rest)
    topics = [w for w in re.findall(r"[a-z0-9\-]+", rest) if w not in STOP and len(w) > 2]
    return {"type": want_type, "sentiment": sentiment, "topics": topics}


def build_index(mentions):
    """mentions: output of Tallie.extract() over many texts. One vote per comment per item."""
    groups, seen = defaultdict(list), set()
    for m in mentions:
        k = _norm(m["text"])
        if (k, m.get("doc")) in seen and m.get("doc") is not None:
            continue
        seen.add((k, m.get("doc")))
        groups[k].append(m)
    index = {}
    for k, ms in groups.items():
        types = Counter(m["type"] for m in ms if m.get("type"))
        t = types.most_common(1)
        index[k] = {"count": len(ms), "polarity": Counter(m["polarity"] for m in ms),
                    "surface": Counter(m["text"] for m in ms),
                    "type": t[0][0] if t else None,
                    "contexts": [(m["text"], m.get("context", "")) for m in ms[:6]]}
    for e in index.values():
        e["name"] = e["surface"].most_common(1)[0][0]
    return index


def answer(query, index, top=15, min_count=1, judge=None, judge_threshold=0.5):
    pq = parse_query(query)
    cands = []
    for e in index.values():
        if e["count"] < min_count:
            continue
        surf = [x for x, _ in e["surface"].most_common(3)]
        if all(len(x.split()) == 1 and x.islower() for x in surf) and e["count"] < 5:
            continue  # stray lower-case words ("aren't", "way")
        if pq["type"] and e["type"] not in TYPE_FAMILY.get(pq["type"], {pq["type"]}):
            continue
        pol = e["polarity"]; n = sum(pol.values()) or 1
        if pq["sentiment"]:
            k = pol.get(pq["sentiment"], 0)
            if k == 0 or k / n < (0.3 if pq["sentiment"] == "negative" else 0.4):
                continue  # 7 negative out of 109 mentions is not "hated"
            score = round(k * k / n, 2)
        else:
            score = e["count"]
        cands.append((e, score))
    if pq["topics"]:
        probs = judge(query, [e for e, _ in cands]) if judge and cands else {}
        cands = [(e, s) for e, s in cands if probs.get(id(e), 0) >= judge_threshold]
    rows = [{"item": e["name"], "type": e["type"], "score": s, "mentions": e["count"],
             "polarity": dict(e["polarity"]), "also_written_as": [x for x, _ in e["surface"].most_common(4)]}
            for e, s in cands]
    rows.sort(key=lambda r: (-r["score"], -r["mentions"]))
    return {"query": query, "parsed": pq, "results": rows[:top]}


def make_judge(model, tok, max_contexts=3, batch_size=32, max_len=256):
    """Model relevance for extra topic words: run the question over a few of the item's comment
    snippets and average the relevance score of that item's span."""
    import torch
    from .data import build_input, collate
    from .model import amp_dtype

    @torch.no_grad()
    def judge(query, entries):
        dev = next(model.parameters()).device
        jobs = [(id(e), name, ctx) for e in entries for name, ctx in e["contexts"][:max_contexts]]
        scores = defaultdict(list)
        for b0 in range(0, len(jobs), batch_size):
            batch = jobs[b0:b0 + batch_size]
            feats, metas = [], []
            for eid, name, ctx in batch:
                ids, offs, qm, tm, base = build_input(tok, query, ctx, max_len)
                feats.append({"input_ids": ids, "query_mask": qm, "bio": [0] * len(ids),
                              "spans": [], "rel": [], "pol": []})
                metas.append((eid, name, ctx, offs, base))
            x = {k: v.to(dev) for k, v in collate(feats, tok.pad_token_id).items()}
            idx = torch.zeros(len(batch), 1, 2, dtype=torch.long, device=dev)
            found = []
            for j, (eid, name, ctx, offs, base) in enumerate(metas):
                i = ctx.find(name)
                toks = [t for t, (s0, e0) in enumerate(offs) if i >= 0 and e0 > i + base and s0 < i + base + len(name)]
                if toks:
                    idx[j, 0, 0], idx[j, 0, 1] = toks[0], toks[-1]
                found.append(bool(toks))
            with torch.autocast("cuda", dtype=amp_dtype(), enabled=dev.type == "cuda"):
                h = model.encode(x["input_ids"], x["attention_mask"])
            rel, _, _ = model.heads(h, x["query_mask"], idx)
            for j, (eid, *_r) in enumerate(metas):
                if found[j]:
                    scores[eid].append(torch.sigmoid(rel[j, 0]).item())
        return {k: sum(v) / len(v) for k, v in scores.items()}
    return judge

In [ ]:
%%writefile /content/repo/scripts/build_dataset.py
"""
Turn teacher labels into training examples for Tallie.

Each example = (query, text, spans). Every mention span is labeled with:
  relevant (0/1) -> does it satisfy THIS query?
  polarity       -> positive / negative / neutral / mixed (the commenter's opinion)

Two sources of queries per chunk:
  1. the teacher's free-form queries (reasoning-heavy, diverse wording)
  2. templated queries computed from the mention labels (exact, cheap, many paraphrases),
     including "negative" queries that match nothing, so the model learns to say "none".

Splits: by THREAD (no leakage). The "games" domain is held out entirely -> test_heldout.jsonl.
"""
import argparse, json, random, re
from collections import defaultdict
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).parent))
from tallie.filters import is_valid_item, find_all

PLURAL = {"book": "books", "movie": "movies", "tv_show": "TV shows", "video_game": "video games",
          "board_game": "board games", "podcast": "podcasts", "music": "albums or songs",
          "software": "software tools", "product": "products", "person": "people (authors, directors, artists)",
          "other": "things"}
POL = {"positive": 0, "negative": 1, "neutral": 2, "mixed": 3}
# several ways people name each type, so the model learns "films" == "movies", "apps" == "software"
TYPE_WORDS = {"book": ["books", "novels", "books", "reads"], "movie": ["movies", "films", "movies"],
              "tv_show": ["TV shows", "TV series", "shows"], "video_game": ["video games", "games", "video games"],
              "board_game": ["board games", "tabletop games"], "podcast": ["podcasts"],
              "music": ["albums", "songs", "bands", "music"], "software": ["software", "tools", "apps", "programs"],
              "product": ["products", "gadgets", "devices"], "person": ["authors", "people", "creators"],
              "other": ["things"]}


def tword(t, rng):
    return rng.choice(TYPE_WORDS.get(t, ["things"]))


def singular(w):
    return w[:-1] if w.endswith("s") else w

T_LIKED = ["{t} people liked", "{t} that were praised", "most loved {t}", "{t} recommended by commenters",
           "which {t} do people enjoy?", "positive mentions of {t}", "{t} people say are great"]
T_HATED = ["{t} people disliked", "most hated {t}", "{t} that were criticized", "{t} people regret",
           "negative opinions about {t}", "which {t} do people think are bad?"]
T_ALL = ["all {t} mentioned", "list every {t_s} that comes up", "{t} discussed here", "every {t_s} named"]
T_ATTR = ["{t} people found {a}", "{a} {t}", "{t} described as {a}", "which {t} are {a}?"]
T_TOPIC = ["{t} related to {p}", "{t} about {p}", "anything about {p}", "{t} on the topic of {p}"]
T_REC = ["things people recommend", "suggestions from commenters", "what should I check out?"]
T_REC_TYPE = ["{t} people recommend", "{t} worth trying according to commenters", "recommended {t}",
              "any good {t}?", "what {t} do people recommend?", "{t} worth checking out"]
T_TOPIC_POS = ["{t} about {p} that people liked", "well-liked {t} related to {p}", "good {t} on {p}"]
T_TOPIC_NEG = ["{t} about {p} that people criticized", "{t} related to {p} people disliked"]

SINGULAR = {"books": "book", "movies": "movie", "TV shows": "TV show", "video games": "video game",
            "board games": "board game", "podcasts": "podcast", "albums or songs": "album or song",
            "software tools": "software tool", "products": "product",
            "people (authors, directors, artists)": "person", "things": "thing"}


NON_LIST_Q = re.compile(r"^(how|why|when|where|is|are there|does|do|did|can|could|should|would|will|who is)\b|steps|requirements|explain", re.I)
STRICT = False  # --strict-names: names must contain a capital letter or digit (for noisy teachers)


def list_query(q):
    """Keep only requests whose answer is a list of items (drop 'how do I...', 'why did...')."""
    return not NON_LIST_Q.search(q.strip())


def name_like(t):
    return is_valid_item(t) and (not STRICT or bool(re.search(r"[A-Z0-9]", t))) and len(t.split()) <= 8


def find_spans(text, mentions):
    """Locate every occurrence of each mention; resolve overlaps (longer wins)."""
    cands = []
    for mi, m in enumerate(mentions):
        if not name_like(m["text"]):   # teacher junk like "books", "it", ",", "wafer"
            continue
        for h in find_all(text, m["text"]):
            cands.append((h.start(), h.end(), mi))
    cands.sort(key=lambda x: (-(x[1] - x[0]), x[0]))
    taken, spans = [], []
    for s, e, mi in cands:
        if any(s < te and ts < e for ts, te in taken):
            continue
        taken.append((s, e))
        spans.append((s, e, mi))
    return sorted(spans)


def make_example(query, text, spans, mentions, relevant_set):
    return {"query": query, "text": text,
            "spans": [{"start": s, "end": e, "text": text[s:e],
                       "relevant": int(mi in relevant_set),
                       "polarity": POL[mentions[mi]["sentiment"]],
                       "type": mentions[mi].get("type", "other")} for s, e, mi in spans]}


def templated(mentions, rng, global_topics, global_attrs, k):
    out = []
    types = sorted({m["type"] for m in mentions})
    all_types = list(PLURAL)
    for _ in range(k):
        kind = rng.choice(["liked", "hated", "all", "attr", "attr", "topic", "topic", "rec", "rec_type", "absent",
                           "topic_pol", "absent"])
        t = rng.choice(types) if types else "book"
        tp = tword(t, rng)
        if kind == "liked":
            q = rng.choice(T_LIKED).format(t=tp)
            rel = {i for i, m in enumerate(mentions) if m["type"] == t and m["sentiment"] == "positive"}
        elif kind == "hated":
            q = rng.choice(T_HATED).format(t=tp)
            rel = {i for i, m in enumerate(mentions) if m["type"] == t and m["sentiment"] == "negative"}
        elif kind == "all":
            q = rng.choice(T_ALL).format(t=tp, t_s=singular(tp))
            rel = {i for i, m in enumerate(mentions) if m["type"] == t}
        elif kind == "attr":
            local = [a for m in mentions if m["type"] == t for a in m["attributes"]]
            a = rng.choice(local) if local and rng.random() < 0.7 else rng.choice(global_attrs)
            q = rng.choice(T_ATTR).format(t=tp, a=a)
            rel = {i for i, m in enumerate(mentions) if m["type"] == t and a in m["attributes"]}
        elif kind == "topic":
            local = [p for m in mentions for p in m["topics"]]
            p = rng.choice(local) if local and rng.random() < 0.7 else rng.choice(global_topics)
            tmpl = rng.choice(T_TOPIC)
            q = tmpl.format(t=tp, p=p)
            anytype = tmpl.startswith("anything")
            rel = {i for i, m in enumerate(mentions)
                   if (anytype or m["type"] == t) and p.lower() in [x.lower() for x in m["topics"]]}
        elif kind == "rec":
            q = rng.choice(T_REC)
            rel = {i for i, m in enumerate(mentions) if m["recommended"]}
        elif kind == "rec_type":
            q = rng.choice(T_REC_TYPE).format(t=tp)
            rel = {i for i, m in enumerate(mentions) if m["type"] == t and m["recommended"]}
        elif kind == "topic_pol":
            local = [(p, m["sentiment"]) for m in mentions if m["type"] == t for p in m["topics"]
                     if m["sentiment"] in ("positive", "negative")]
            if not local:
                continue
            p, sent = rng.choice(local)
            q = rng.choice(T_TOPIC_POS if sent == "positive" else T_TOPIC_NEG).format(t=tp, p=p)
            rel = {i for i, m in enumerate(mentions) if m["type"] == t and m["sentiment"] == sent
                   and p.lower() in [x.lower() for x in m["topics"]]}
        else:  # a type that is not in this chunk -> nothing relevant
            absent = [x for x in all_types if x not in types and x != "other"]
            if not absent:
                continue
            w = tword(rng.choice(absent), rng)
            q = rng.choice(T_ALL + T_LIKED + T_HATED + T_REC_TYPE).format(t=w, t_s=singular(w))
            rel = set()
        if not rel and kind != "absent" and rng.random() < 0.5:
            continue  # keep empty answers mostly from deliberate "wrong type" questions
        out.append((q, rel))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--inp", default="data/labeled/labels.jsonl")
    ap.add_argument("--outdir", default="data/processed")
    ap.add_argument("--heldout-domain", default="games")
    ap.add_argument("--templated-per-chunk", type=int, default=2)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--strict-names", action="store_true")
    args = ap.parse_args()
    global STRICT
    STRICT = args.strict_names
    rng = random.Random(args.seed)

    rows = [json.loads(l) for l in open(args.inp, encoding="utf-8")]
    global_topics = sorted({p for r in rows for m in r["mentions"] for p in m["topics"]}) or ["space"]
    global_attrs = sorted({a for r in rows for m in r["mentions"] for a in m["attributes"]}) or ["funny"]

    threads = sorted({r["thread_id"] for r in rows if r["domain"] != args.heldout_domain})
    rng.shuffle(threads)
    n = len(threads)
    split_of = {t: "train" for t in threads[: int(0.9 * n)]}
    split_of.update({t: "val" for t in threads[int(0.9 * n): int(0.95 * n)]})
    split_of.update({t: "test" for t in threads[int(0.95 * n):]})

    buckets = defaultdict(list)
    for r in rows:
        split = "test_heldout" if r["domain"] == args.heldout_domain else split_of[r["thread_id"]]
        spans = find_spans(r["text"], r["mentions"])
        mentions = r["mentions"]
        queries = [(q["query"], set(q["relevant"])) for q in r["queries"] if list_query(q["query"])]
        if mentions:
            k = args.templated_per_chunk + (2 if not queries else 0)  # more templated when no teacher queries
            queries += templated(mentions, rng, global_topics, global_attrs, k)
        elif rng.random() < 0.3:  # some mention-free chunks so the model learns to output nothing
            queries += templated([], rng, global_topics, global_attrs, 1)
        for q, rel in queries:
            ex = make_example(q, r["text"], spans, mentions, rel)
            ex.update({"chunk_id": r["chunk_id"], "thread_id": r["thread_id"], "domain": r["domain"]})
            buckets[split].append(ex)

    Path(args.outdir).mkdir(parents=True, exist_ok=True)
    for split, exs in buckets.items():
        rng.shuffle(exs)
        with open(Path(args.outdir) / f"{split}.jsonl", "w", encoding="utf-8") as f:
            for ex in exs:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
        npos = sum(s["relevant"] for ex in exs for s in ex["spans"])
        print(f"{split:13s} {len(exs):7d} examples, {npos} relevant spans")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/repo/scripts/train.py
"""
Train Tallie.

  python train.py --data data/processed --backbone Qwen/Qwen3-1.7B --keep-layers 20 --out runs/tallie
"""
import argparse, json, math, sys, time
from functools import partial
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).parent))
from tallie.model import TallieModel, amp_dtype
from tallie.data import TallieDataset, collate
from tallie.infer import save


@torch.no_grad()
def evaluate(model, loader, device, max_batches=0, thr=0.5):
    model.eval()
    import math
    logit_thr = math.log(thr / (1 - thr))
    typ_ok = typ_n = 0
    tp = fp = fn = 0            # mention spans
    e2e_tp = e2e_fp = e2e_fn = 0  # end-to-end: predicted span AND judged relevant
    rel_ok = rel_n = pol_ok = pol_n = 0
    for bi, x in enumerate(loader):
        if max_batches and bi >= max_batches:
            break
        x = {k: v.to(device) for k, v in x.items()}
        with torch.autocast("cuda", dtype=amp_dtype(), enabled=device.type == "cuda"):
            out = model(**{k: x[k] for k in ("input_ids", "attention_mask", "query_mask",
                                              "span_idx", "span_valid")})
        v = x["span_valid"].bool()
        rel_pred = out["rel_logits"] > 0
        rel_ok += (rel_pred[v] == x["rel_labels"][v].bool()).sum().item(); rel_n += v.sum().item()
        pol_ok += (out["pol_logits"].argmax(-1)[v] == x["pol_labels"][v]).sum().item(); pol_n += v.sum().item()
        tv = v & (x["type_labels"] >= 0)
        typ_ok += (out["type_logits"].argmax(-1)[tv] == x["type_labels"][tv]).sum().item(); typ_n += tv.sum().item()

        tags = out["bio_logits"].argmax(-1).tolist()
        text_mask = (x["bio_labels"] != -100) & (x["query_mask"] == 0)
        with torch.autocast("cuda", dtype=amp_dtype(), enabled=device.type == "cuda"):
            h = model.encode(x["input_ids"], x["attention_mask"])
        for i in range(len(tags)):
            gold = {tuple(s) for s, ok in zip(x["span_idx"][i].tolist(), v[i].tolist()) if ok}
            gold_rel = {tuple(s) for s, ok, r in zip(x["span_idx"][i].tolist(), v[i].tolist(),
                                                      x["rel_labels"][i].tolist()) if ok and r}
            pred = set(model.decode_bio(tags[i], text_mask[i].tolist()))
            tp += len(pred & gold); fp += len(pred - gold); fn += len(gold - pred)
            if pred:
                pl = sorted(pred)
                idx = torch.tensor([pl], device=device)
                r, _, _ = model.heads(h[i:i + 1], x["query_mask"][i:i + 1], idx)
                pred_rel = {s for s, sc in zip(pl, r[0].tolist()) if sc > logit_thr}
            else:
                pred_rel = set()
            e2e_tp += len(pred_rel & gold_rel); e2e_fp += len(pred_rel - gold_rel); e2e_fn += len(gold_rel - pred_rel)
    f1 = lambda a, b, c: 2 * a / max(1, 2 * a + b + c)
    model.train()
    return {"mention_f1": round(f1(tp, fp, fn), 4),
            "relevance_acc": round(rel_ok / max(1, rel_n), 4),
            "polarity_acc": round(pol_ok / max(1, pol_n), 4),
            "type_acc": round(typ_ok / max(1, typ_n), 4),
            "end_to_end_f1": round(f1(e2e_tp, e2e_fp, e2e_fn), 4)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/processed")
    ap.add_argument("--backbone", default="Qwen/Qwen3-1.7B")
    ap.add_argument("--keep-layers", type=int, default=20)
    ap.add_argument("--causal", action="store_true", help="ablation: keep causal attention")
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--bio-weight", type=float, default=3.0, help="extra weight on name tokens")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--grad-accum", type=int, default=1)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--head-lr", type=float, default=1e-3)
    ap.add_argument("--max-len", type=int, default=384)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--eval-every", type=int, default=500)
    ap.add_argument("--out", default="runs/tallie")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(args.backbone)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    # balanced opinion training: inverse-frequency weights (labels lean heavily positive)
    from collections import Counter
    pc = Counter(sp["polarity"] for l in open(Path(args.data) / "train.jsonl", encoding="utf-8")
                 for sp in json.loads(l)["spans"])
    tot = sum(pc.values()) or 1
    pol_w = [min(4.0, (tot / (4 * max(1, pc.get(i, 0)))) ** 0.5) for i in range(4)]
    print("polarity counts", dict(pc), "-> loss weights", [round(w, 2) for w in pol_w])
    model = TallieModel(args.backbone, keep_layers=args.keep_layers,
                         bidirectional=not args.causal, lora_r=args.lora_r,
                         bio_weight=args.bio_weight, pol_class_weights=pol_w).to(device)
    print(json.dumps(model.param_report(), indent=2))

    D = Path(args.data)
    train = TallieDataset(D / "train.jsonl", tok, args.max_len, args.limit)
    val = TallieDataset(D / "val.jsonl", tok, args.max_len)
    col = partial(collate, pad_id=tok.pad_token_id)
    vl = DataLoader(val, batch_size=args.batch_size, collate_fn=col)

    lora = [p for n, p in model.named_parameters() if n.startswith("backbone.") and p.requires_grad]
    opt = torch.optim.AdamW([{"params": lora, "lr": args.lr},
                             {"params": model.head_params(), "lr": args.head_lr}], weight_decay=0.01)
    batches_per_epoch = math.ceil(len(train) / args.batch_size)
    steps = math.ceil(batches_per_epoch / args.grad_accum) * args.epochs
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * steps), steps)
    adt = amp_dtype()
    scaler = torch.amp.GradScaler("cuda", enabled=adt == torch.float16)
    print("mixed precision:", adt)

    trainable = lambda: {k: v for k, v in model.state_dict().items()
                         if "lora_" in k or not k.startswith("backbone.")}
    best, step, start_ep, skip, t0 = -1, 0, 0, 0, time.time()
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    last_path = Path(args.out + "_last.pt")
    if last_path.exists():  # resume after a Colab disconnect
        ck = torch.load(last_path, map_location=device)
        model.load_state_dict(ck["model"], strict=False)
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        best, step = ck["best"], ck["step"]
        start_ep, skip = divmod(step * args.grad_accum, batches_per_epoch)
        print(f"resumed at step {step} (epoch {start_ep}, batch {skip})")
    log = open(args.out + ".log", "a")
    model.train()
    for ep in range(start_ep, args.epochs):
        g = torch.Generator().manual_seed(1000 + ep)  # same order on resume
        tl = DataLoader(train, batch_size=args.batch_size, shuffle=True, generator=g,
                        collate_fn=col, num_workers=2)
        for bi, x in enumerate(tl):
            if ep == start_ep and bi < skip:
                continue
            x = {k: v.to(device) for k, v in x.items()}
            with torch.autocast("cuda", dtype=adt, enabled=device.type == "cuda"):
                out = model(**x)
            scaler.scale(out["loss"] / args.grad_accum).backward()
            if (bi + 1) % args.grad_accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % 50 == 0:
                msg = (f"ep {ep} step {step}/{steps} loss {out['loss'].item():.3f} "
                       f"(bio {out['l_bio']:.3f} rel {out['l_rel']:.3f} pol {out['l_pol']:.3f}) "
                       f"{(time.time() - t0) / 60:.1f} min")
                print(msg); log.write(msg + "\n"); log.flush()
            if step % args.eval_every == 0 or step == steps:
                m = evaluate(model, vl, device, max_batches=100)
                print("VAL", m); log.write(f"VAL step {step} {json.dumps(m)}\n"); log.flush()
                if m["end_to_end_f1"] > best:
                    best = m["end_to_end_f1"]
                    torch.save(trainable(), Path(args.out + "_best_trainable.pt"))
                torch.save({"model": trainable(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                            "scaler": scaler.state_dict(), "best": best, "step": step}, last_path)
    # load best trainable weights back, then merge LoRA and save the final model
    best_path = Path(args.out + "_best_trainable.pt")
    if best_path.exists():
        model.load_state_dict(torch.load(best_path, map_location=device), strict=False)
    # pick the relevance cutoff that maximizes end-to-end F1 on the validation set
    best_thr, final = 0.5, None
    for thr in (0.5, 0.3, 0.35, 0.4, 0.45, 0.55, 0.6, 0.65):  # 0.5 first: ties keep the default
        m = evaluate(model, vl, device, thr=thr)
        print(f"threshold {thr:.2f}: {m}")
        if final is None or m["end_to_end_f1"] > final["end_to_end_f1"]:
            best_thr, final = thr, m
    final["threshold"] = best_thr
    print("FINAL VAL", final)
    save(model, tok, args.out, extra={"threshold": best_thr})
    (Path(args.out) / "val_metrics.json").write_text(json.dumps(final, indent=2))
    print("saved ->", args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/repo/scripts/evaluate.py
"""
Evaluate a saved Tallie model on test splits and measure speed.

  python evaluate.py --ckpt runs/tallie --data data/processed
"""
import argparse, json, sys, time
from functools import partial
from pathlib import Path

import torch
from torch.utils.data import DataLoader

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).parent))
from tallie.data import TallieDataset, collate
from tallie.infer import load, extract
from train import evaluate


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default="runs/tallie")
    ap.add_argument("--data", default="data/processed")
    ap.add_argument("--max-len", type=int, default=384)
    args = ap.parse_args()
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, tok = load(args.ckpt, device=dev)
    report = {"params": model.param_report()}
    for split in ("test", "test_heldout"):
        p = Path(args.data) / f"{split}.jsonl"
        if p.exists():
            dl = DataLoader(TallieDataset(p, tok, args.max_len), batch_size=16,
                            collate_fn=partial(collate, pad_id=tok.pad_token_id))
            report[split] = evaluate(model, dl, dev)
            print(split, report[split])

    # throughput: comments per second for one query
    rows = [json.loads(l) for l in open(Path(args.data) / "test.jsonl", encoding="utf-8")][:512]
    texts = [r["text"] for r in rows]
    extract(model, tok, "books people liked", texts[:32])  # warm-up
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    extract(model, tok, "books people liked", texts, batch_size=32)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    report["comments_per_second"] = round(len(texts) / (time.time() - t), 1)
    print("throughput:", report["comments_per_second"], "comments/s")
    (Path(args.ckpt) / "test_report.json").write_text(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/repo/scripts/eval_gold.py
"""
Score models (and the teacher) against the HAND-LABELED test set (data/gold_test.jsonl).
This is the trustworthy number; the old test split is scored against noisy teacher labels.

  python eval_gold.py --gold data/gold_test.jsonl --ckpt runs/tallie \
                      --teacher data/labeled/labels.jsonl
"""
import argparse, json, sys
from collections import defaultdict
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).parent))
from tallie.data import build_input, collate
from tallie.filters import clean_span, is_valid_item
from tallie.infer import load, normalize
from tallie.model import amp_dtype, POLARITY, TYPES


def norm(t):
    return normalize(t).replace("’", "'")


def f1(tp, fp, fn):
    p = tp / max(1, tp + fp); r = tp / max(1, tp + fn)
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(2 * p * r / max(1e-9, p + r), 3)}


@torch.no_grad()
def predict(model, tok, query, text, max_len=384):
    """All predicted mentions with (relevance score, polarity) for one (query, text)."""
    dev = next(model.parameters()).device
    ids, offs, qm, tm, base = build_input(tok, query, text, max_len)
    x = collate([{"input_ids": ids, "query_mask": qm, "bio": [0] * len(ids), "spans": [], "rel": [], "pol": []}],
                tok.pad_token_id)
    x = {k: v.to(dev) for k, v in x.items()}
    with torch.autocast("cuda", dtype=amp_dtype(), enabled=dev.type == "cuda"):
        h = model.encode(x["input_ids"], x["attention_mask"])
    tags = model.mention_head(h).argmax(-1)[0].tolist()
    spans = model.decode_bio(tags, tm)
    if not spans:
        return []
    rel, pol, typ = model.heads(h, x["query_mask"], torch.tensor([spans], device=dev))
    out = []
    for (a, b), r, p, t in zip(spans, torch.sigmoid(rel[0]).tolist(), pol[0].argmax(-1).tolist(),
                               typ[0].argmax(-1).tolist()):
        fx = clean_span(text, max(offs[a][0] - base, 0), offs[b][1] - base)
        if fx and is_valid_item(text[fx[0]:fx[1]].strip("\"'“”‘’")):
            out.append({"text": text[fx[0]:fx[1]].strip("\"'“”‘’"), "score": r, "polarity": POLARITY[p],
                        "type": TYPES[t]})
    return out


def score_model(model, tok, gold, threshold):
    S = defaultdict(lambda: defaultdict(int))
    examples = []
    for g in gold:
        dom = "games (held out)" if g["domain"] == "games" else "seen domains"
        gm = {norm(m["text"]): m for m in g["mentions"]}
        preds_any = None
        for q in g["queries"]:
            preds = predict(model, tok, q["query"], g["text"])
            if preds_any is None:  # names found (query-independent part), scored once per comment
                preds_any = preds
                pm = {norm(p["text"]): p for p in preds}
                for k in ("all", dom):
                    S[k]["m_tp"] += len(set(pm) & set(gm)); S[k]["m_fp"] += len(set(pm) - set(gm))
                    S[k]["m_fn"] += len(set(gm) - set(pm))
                    for n in set(pm) & set(gm):
                        S[k]["pol_n"] += 1; S[k]["pol_ok"] += int(pm[n]["polarity"] == gm[n]["sentiment"])
                        S[k]["typ_n"] += 1; S[k]["typ_ok"] += int(pm[n].get("type") == gm[n].get("type"))
            pr = {norm(p["text"]) for p in preds if p["score"] >= threshold}
            gr = {norm(t) for t in q["relevant"]}
            for k in ("all", dom):
                S[k]["e_tp"] += len(pr & gr); S[k]["e_fp"] += len(pr - gr); S[k]["e_fn"] += len(gr - pr)
            if len(examples) < 40 and (pr != gr):
                examples.append({"query": q["query"], "gold": sorted(gr), "model": sorted(pr),
                                 "text": g["text"][:300]})
    rep = {}
    for k, s in S.items():
        rep[k] = {"names_found": f1(s["m_tp"], s["m_fp"], s["m_fn"]),
                  "end_to_end": f1(s["e_tp"], s["e_fp"], s["e_fn"]),
                  "opinion_accuracy": round(s["pol_ok"] / max(1, s["pol_n"]), 3),
                  "type_accuracy": round(s["typ_ok"] / max(1, s["typ_n"]), 3)}
    return rep, examples


def score_teacher(gold, labels_path):
    lab = {}
    for l in open(labels_path, encoding="utf-8"):
        r = json.loads(l); lab[r["chunk_id"]] = r
    S = defaultdict(lambda: defaultdict(int)); n = 0
    for g in gold:
        t = lab.get(g["chunk_id"])
        if t is None:
            continue
        n += 1
        dom = "games (held out)" if g["domain"] == "games" else "seen domains"
        gm = {norm(m["text"]): m for m in g["mentions"]}
        tm = {norm(m["text"]): m for m in t["mentions"]}
        for k in ("all", dom):
            S[k]["tp"] += len(set(tm) & set(gm)); S[k]["fp"] += len(set(tm) - set(gm)); S[k]["fn"] += len(set(gm) - set(tm))
            for x in set(tm) & set(gm):
                S[k]["pn"] += 1; S[k]["pok"] += int(tm[x]["sentiment"] == gm[x]["sentiment"])
    return {"comments_covered": n,
            **{k: {"names_found": f1(s["tp"], s["fp"], s["fn"]),
                   "opinion_accuracy": round(s["pok"] / max(1, s["pn"]), 3)} for k, s in S.items()}}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gold", default="data/gold_test.jsonl")
    ap.add_argument("--ckpt", nargs="*", default=[])
    ap.add_argument("--teacher", nargs="*", default=[], help="labels.jsonl files to score as teachers")
    ap.add_argument("--threshold", type=float, default=0.5)
    ap.add_argument("--out", default="gold_report.json")
    a = ap.parse_args()
    gold = [json.loads(l) for l in open(a.gold, encoding="utf-8")]
    report = {}
    for t in a.teacher:
        report[f"teacher: {t}"] = score_teacher(gold, t)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    for c in a.ckpt:
        if not Path(c).exists():
            print("skip (not found):", c); continue
        model, tok = load(c, device=dev); model.eval()
        rep, ex = score_model(model, tok, gold, a.threshold)
        report[f"model: {c}"] = rep
        report[f"model: {c} | example mistakes"] = ex
        del model; torch.cuda.empty_cache() if dev == "cuda" else None
    json.dump(report, open(a.out, "w"), indent=1, ensure_ascii=False)
    for k, v in report.items():
        if "example mistakes" in k:
            continue
        print("\n###", k)
        print(json.dumps(v, indent=1))
    print("\nfull report ->", a.out)




if __name__ == "__main__":
    main()

In [ ]:
REPO = "/content/repo"; sys.path.insert(0, REPO); sys.path.insert(0, f"{REPO}/scripts")
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH, ACCUM, MAX_LEN = (16, 1, 384) if mem > 38 else (8, 2, 384) if mem > 20 else (8, 2, 256)
EPOCHS, LAYERS = 3, 20
RUN = "runs/tallie"
print(f"{torch.cuda.get_device_name(0)} ({mem:.0f} GB): batch {BATCH}x{ACCUM}, max_len {MAX_LEN}, {EPOCHS} epochs -> {RUN}")

## 1. Build the training set

In [ ]:
!python {REPO}/scripts/build_dataset.py --inp data/labeled/labels_clean.jsonl --outdir data/processed
import collections
rows = [json.loads(l) for l in open("data/processed/train.jsonl")]
print("empty-answer share:", round(sum(1 for r in rows if not any(s["relevant"] for s in r["spans"])) / len(rows), 2))
print("types:", collections.Counter(s["type"] for r in rows for s in r["spans"]).most_common())

## 2. Train (~40 min on an L4; re-run the cell to resume after a disconnect)

In [ ]:
!python {REPO}/scripts/train.py --data data/processed --backbone Qwen/Qwen3-1.7B --keep-layers {LAYERS} --epochs {EPOCHS} --batch-size {BATCH} --grad-accum {ACCUM} --max-len {MAX_LEN} --eval-every 300 --out {RUN}
!ls -la {RUN}/backbone   # model.safetensors should be ~2.5 GB

## 3. Test split, held-out games topic, speed

In [ ]:
!python {REPO}/scripts/evaluate.py --ckpt {RUN} --data data/processed --max-len {MAX_LEN}

## 4. Hand-labeled test set

In [ ]:
!python {REPO}/scripts/eval_gold.py --gold data/gold_test.jsonl --ckpt {RUN}